# אימון גלאי אבטחה (security.pt): אקדח, סכין, אש, עשן — Google Colab

לפני שמריצים: **Runtime → Change runtime type → T4 GPU**.

המחברת מורידה את שני הדאטהסטים ישירות לקולאב (אקדחים 1.5 GB ממצלמות אבטחה אמיתיות, D-Fire 3.1 GB), בונה דאטהסט אחד, מאמנת YOLOv8s ומורידה `best.pt`.

מומלץ טוקן של Hugging Face ב-Secrets (`HF_TOKEN`, Notebook access דלוק), אחרת ההורדה עלולה להיחסם זמנית.

In [ ]:
# 1. setup
!pip -q install ultralytics huggingface_hub pyarrow
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — switch runtime to GPU')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF token: ok')
except Exception:
    print('no HF_TOKEN secret — continuing without it')

In [ ]:
# 2. the dataset scripts from the repo
import urllib.request, sys
BASE = 'https://raw.githubusercontent.com/Talyaavraham/final_project/main/backend/train/'
for f in ('night_dataset.py', 'security_dataset.py'):
    urllib.request.urlretrieve(BASE + f, '/content/' + f); print('got', f)
sys.path.insert(0, '/content')

In [ ]:
# 3. download the sources (~4.6 GB; a few minutes on Colab)
from huggingface_hub import hf_hub_download, snapshot_download
os.makedirs('/content/backend/eval/data/weapons', exist_ok=True)
hf_hub_download('jsalazar/US-Real-time-gun-detection-in-CCTV-An-open-problem-dataset', 'weapons_images_2fps.zip',
                repo_type='dataset', local_dir='/content/backend/eval/data/weapons')
snapshot_download('badsaarow/d-fire', repo_type='dataset', local_dir='/content/backend/train/security/dfire',
                  allow_patterns=['data/*.parquet'])
print('sources ready')

In [ ]:
# 4. (optional) your own labelled frames: a Roboflow "YOLOv8" export zipped. Press Cancel to skip.
from google.colab import files
import shutil, zipfile
own = []
up = files.upload()
for name in up:
    shutil.rmtree('/content/own', ignore_errors=True)
    zipfile.ZipFile(name).extractall('/content/own')
    root = '/content/own'
    if not os.path.exists(root + '/data.yaml'):
        subs = [d for d in os.listdir(root) if os.path.isdir(f'{root}/{d}')]
        root = f'{root}/{subs[0]}' if subs else root
    own = [root]; print('own export:', root)
print('own exports:', own or 'none')

In [ ]:
# 5. build the dataset (classes: gun, knife, fire, smoke)
import security_dataset as sd
sd.WEAPONS_ZIP = '/content/backend/eval/data/weapons/weapons_images_2fps.zip'
sd.DFIRE_DIR = '/content/backend/train/security/dfire/data'
w = sd.Writer('/content/security/dataset')
sd.add_weapons(w); sd.add_dfire(w)
for d in own: sd.add_own(w, d)
w.yaml(); print(w.n, w.boxes)

In [ ]:
# 6. train. yolov8s, imgsz 800: ~3 min/epoch on a T4 -> 25 epochs ≈ 1.5 h. In a hurry: epochs=12.
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data='/content/security/dataset/data.yaml', epochs=25, imgsz=800, batch=16, device=0,
            project='/content/runs', name='security', exist_ok=True, seed=0,
            close_mosaic=5, scale=0.5, patience=8)

In [ ]:
# 7. metrics per class on the validation split
m = YOLO('/content/runs/security/weights/best.pt')
r = m.val(data='/content/security/dataset/data.yaml', imgsz=800, plots=False)
for i, n in r.names.items():
    print(f'{n:8s} mAP50={r.box.ap50[i]:.3f}  mAP50-95={r.box.maps[i]:.3f}')
print('all      mAP50=%.3f' % r.box.map50)

In [ ]:
# 8. download best.pt  ->  save it as backend/security.pt
files.download('/content/runs/security/weights/best.pt')

## חזרה למחשב

```powershell
copy best.pt backend\security.pt
cd backend
$env:YOLO_MODEL = "yolov8n.pt,game.pt,night.pt"
$env:EXTRA_MODELS = "security.pt"
$env:IMGSZ = "800"
.env\Scripts\python -m uvicorn main:app --port 8000
```

`EXTRA_MODELS` = מודלים שרצים **בנוסף** למודל שנבחר לסרטון (הגלאי הראשי מוצא אנשים ורכבים, security.pt מוצא נשק ואש באותם פריימים).